# ANN (FNN)

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv("powerplant_data.csv")
df.head()

In [ ]:
# AT => temperature
# V  => vacuum
# AP => pressure
# RH => humidity

# PE => produced energy

In [ ]:
df.isnull().sum()

In [ ]:
X = df.drop(columns = "PE")
y = df["PE"]

In [ ]:
X.head()

In [ ]:
y.head()

In [ ]:
# Split our data
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
df.shape


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scale = scaler.fit_transform(X_train)
X_test_scale = scaler.transform(X_test)

In [ ]:
X_train_scale

In [ ]:
X_test_scale

In [ ]:
import torch 
import torch.nn as nn

In [ ]:
# Creating tensors

X_train_tensor = torch.tensor(X_train_scale, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).view(-1,1)  # .values because y_train is a pandas series and .view check notes 

X_test_tensor = torch.tensor(X_test_scale, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).view(-1,1)

# And these tensors we created are being stored inside our PC - RAM

In [ ]:
type(X_train_scale)

In [ ]:
type(y_train)

In [ ]:
y_train.shape

In [ ]:
# Creating tensordataset and dataloader

from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=32)

# Deep Learning starts from here

In [ ]:
# Defining our model

class FNN(nn.Module):
    def __init__(self):
        super(FNN, self).__init__()

        self.model = nn.Sequential(
            # 1st hidden layer
            nn.Linear(X_train.shape[1], 6), # in_features, out_features as params
            nn.ReLU(),

            # 2nd hidden layer
            nn.Linear(6, 6),  # in_features, out_features as params
            nn.ReLU(),

            # o/p layer
            nn.Linear(6, 1),  # in_features, out_features as params
        )
    # defining forward prop
    def forward(self, x):
        return self.model(x)
    # we need not have to specify fnx for backward prop because it is inbuilt in torch

In [ ]:
import torch.optim as optim

model = FNN()

# loss, optimzer
crietrion = nn.MSELoss()
optimizer = optim.Adam(model.parameters()) # params are nothing but the weights and bias etc...

In [ ]:
# Train our ANN (Check lecture to understand the flow)

train_losses = []
val_losses = []
best_val_loss = float("inf") # init it with inf so that the one with the minimal will be the best val loss (Check if condition below)
epochs = 50


for epoch in range(epochs):
    # Training
    model.train() # Entered training mode where forward and backward prop.. is carried out
    running_loss = 0.0 # tot training loss for 1 epoch

    for xb, yb in train_dataloader: # b is batch
        # xb = actual i/p of 1 batch
        # yb = actual o/p of 1 batch
        optimizer.zero_grad() # for fresh gradient values everytime
        
        outputs = model(xb) # forward prop... is carried out and we get the predicted o/p for this batch
        loss = crietrion(outputs, yb) # computes the loss by taking the actual o/p and the predicted o/p
        loss.backward() # back prop... gradients are computed
        optimizer.step() # params are updated

        running_loss += loss.item() # loss is a tensor value so we are converting it to python float

    epoch_train_loss = running_loss / len(train_dataloader) # formula check notes
    train_losses.append(epoch_train_loss)

    # Validation (here their is no learning, only predictions)
    model.eval() # Entered validation mode where only forward is present and no backward propagation
    running_loss = 0.0

    with torch.no_grad(): # Dont compute gradients because it is val here there is no backward prop...
        for xb, yb in test_dataloader:
            outputs = model(xb) # forward prop... 
            loss = crietrion(outputs, yb)
    
            running_loss += loss

    epoch_val_loss = running_loss / len(test_dataloader)
    val_losses.append(epoch_val_loss)

    print(f"epoch: {epoch+1} / {epochs} => train loss = {epoch_train_loss} & val loss = {epoch_val_loss}")

    # Saving our best model
    if epoch_val_loss < best_val_loss :
        best_val_loss = epoch_val_loss

        torch.save(model.state_dict(), "best_model.pt") # .pt and .pth are the extensions to store the file

In [ ]:
# Loading the model
model.load_state_dict(torch.load("best_model.pt"))

In [ ]:
import matplotlib.pyplot as plt

loss_df = pd.DataFrame({
    "Training Loss": train_losses,
    "Validation Loss":val_losses
})

plt.plot(loss_df["Training Loss"], label = "Training Loss")
plt.plot(loss_df["Validation Loss"], label = "Validation Loss")

plt.xlabel("Epochs")
plt.ylabel("Losses")

plt.legend()

In [ ]:
# Evaluation
model.eval()
with torch.no_grad():
    train_pred = model(X_train_tensor)
    test_pred = model(X_test_tensor)

    train_mse_loss = crietrion(train_pred, y_train_tensor)
    test_mse_loss = crietrion(test_pred, y_test_tensor)

print("Training MSE:", train_mse_loss.item())
print("Testing MSE:", test_mse_loss.item())

In [ ]:
from sklearn.metrics import r2_score

print("r2 score:", r2_score(y_test, test_pred))

In [ ]:
# Checking actual values and the predicted values.
actual_df = pd.DataFrame(y_test.values, columns=["Actual Values"])
predicted_df = pd.DataFrame(test_pred.numpy(), columns=["Predicted Values"])

pd.concat([actual_df, predicted_df], axis=1)